# 🎙️ Kokoro-82M Text-to-Speech API Server (Colab + Cloudflared)
Notebook này khởi chạy **FastAPI Server** sử dụng model **Kokoro-82M** (TTS Tiếng Anh chất lượng phòng thu, siêu tự nhiên) và mở Public HTTPS Tunnel qua Cloudflare Tunnel để kết nối với Web Client / Node.js.

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!apt-get -q -y install espeak-ng > /dev/null 2>&1
!pip install -q kokoro>=0.3.4 soundfile fastapi uvicorn pydantic pycloudflared nest-asyncio

In [ ]:
# 2. Khởi tạo FastAPI Server với Kokoro-82M
import io
import os
import soundfile as sf
import torch
from fastapi import FastAPI, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel
from kokoro import KPipeline

# Kiểm tra GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Loading Kokoro Pipeline on device: {device}...")

# Khởi tạo KPipeline cho tiếng Anh ('a' = American English, 'b' = British English)
pipeline_us = KPipeline(lang_code='a', device=device)
pipeline_uk = KPipeline(lang_code='b', device=device)

app = FastAPI(title="Kokoro-82M TTS API", description="FastAPI Backend for English Text-to-Speech")

# Bật CORS để cho phép Web Client gọi API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

VOICE_CATALOG = [
    {"id": "af_heart", "name": "Heart (American Female - Premium)", "lang": "en-US", "gender": "female"},
    {"id": "af_bella", "name": "Bella (American Female)", "lang": "en-US", "gender": "female"},
    {"id": "af_sarah", "name": "Sarah (American Female)", "lang": "en-US", "gender": "female"},
    {"id": "af_nicole", "name": "Nicole (American Female)", "lang": "en-US", "gender": "female"},
    {"id": "af_sky", "name": "Sky (American Female)", "lang": "en-US", "gender": "female"},
    {"id": "af_alloy", "name": "Alloy (American Female)", "lang": "en-US", "gender": "female"},
    {"id": "af_jessica", "name": "Jessica (American Female)", "lang": "en-US", "gender": "female"},
    {"id": "am_adam", "name": "Adam (American Male)", "lang": "en-US", "gender": "male"},
    {"id": "am_michael", "name": "Michael (American Male)", "lang": "en-US", "gender": "male"},
    {"id": "am_eric", "name": "Eric (American Male)", "lang": "en-US", "gender": "male"},
    {"id": "am_liam", "name": "Liam (American Male)", "lang": "en-US", "gender": "male"},
    {"id": "am_onyx", "name": "Onyx (American Male)", "lang": "en-US", "gender": "male"},
    {"id": "bf_emma", "name": "Emma (British Female)", "lang": "en-GB", "gender": "female"},
    {"id": "bf_isabella", "name": "Isabella (British Female)", "lang": "en-GB", "gender": "female"},
    {"id": "bm_george", "name": "George (British Male)", "lang": "en-GB", "gender": "male"},
    {"id": "bm_lewis", "name": "Lewis (British Male)", "lang": "en-GB", "gender": "male"}
]

class TTSRequest(BaseModel):
    text: str
    voice: str = "af_heart"
    speed: float = 1.0

@app.get("/")
def read_root():
    return {"status": "ok", "service": "Kokoro-82M TTS Server", "model_sample_rate": 24000}

@app.get("/speakers")
def get_speakers():
    return VOICE_CATALOG

def generate_audio(text: str, voice: str = "af_heart", speed: float = 1.0):
    if not text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty")
    
    # Chọn pipeline phù hợp theo prefix (b = British, a = American)
    selected_pipeline = pipeline_uk if voice.startswith("b") else pipeline_us
    
    generator = selected_pipeline(text, voice=voice, speed=speed, split_pattern=r'\n+')
    audio_segments = []
    
    for _, _, audio in generator:
        audio_segments.append(audio)
        
    if not audio_segments:
        raise HTTPException(status_code=500, detail="Failed to generate audio segments")
        
    import numpy as np
    full_audio = np.concatenate(audio_segments)
    
    buffer = io.BytesIO()
    sf.write(buffer, full_audio, 24000, format='WAV')
    buffer.seek(0)
    return buffer.getvalue()

@app.post("/tts")
def tts_post(req: TTSRequest):
    audio_bytes = generate_audio(req.text, req.voice, req.speed)
    return Response(content=audio_bytes, media_type="audio/wav", headers={"Content-Disposition": "attachment; filename=output.wav"})

@app.get("/tts")
def tts_get(text: str = Query(...), voice: str = Query("af_heart"), speed: float = Query(1.0)):
    audio_bytes = generate_audio(text, voice, speed)
    return Response(content=audio_bytes, media_type="audio/wav", headers={"Content-Disposition": "attachment; filename=output.wav"})


In [ ]:
# 3. Khởi chạy Uvicorn Server và mở Public Tunnel (Cloudflared)
import subprocess
import threading
import time
import uvicorn
import re

# Chạy FastAPI qua Uvicorn trên luồng nền (Port 8000)
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(2)

print("🟢 FastAPI Server is running on port 8000!")
print("⏳ Starting Cloudflared Tunnel...")

# Tải và chạy binary cloudflared trực tiếp để lấy public URL ổn định
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

tunnel_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in tunnel_process.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print("\n" + "="*60)
        print(f"🎉 SUCCESS! YOUR PUBLIC API URL IS:")
        print(f"👉 {public_url} 👈")
        print("="*60)
        print("Copy đường link trên và dán vào file config.js trong dự án Node.js của bạn!\n")
        break

# Giữ cell chạy liên tục để duy trì tunnel
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Tunnel stopped.")